In [0]:
import json
from logging import config
import os
import random
import uuid as py_uuid
import time
from datetime import datetime, timezone
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
def load_properties(filepath):
    props = {}
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            key, value = line.split("=", 1)
            props[key.strip()] = value.strip()

    return props

config = load_properties("/Workspace/Users/rcoundinya@gmail.com/ride-analytics-platform/client.properties")

KAFKA_BROKER = config["bootstrap.servers"]
USER_NAME = config["sasl.username"]
PWD = config["sasl.password"]
TOPIC = config["sasl.topic"]

In [0]:
df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKER)
        .option("kafka.security.protocol", "SASL_SSL") \
        .option("kafka.sasl.mechanism", "PLAIN") \
        .option("kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{USER_NAME}" password="{PWD}";')
        .option("subscribe", TOPIC)
        .option("startingOffsets", "earliest")
        .option("maxOffsetsPerTrigger", "10000")
        .load()
)

parsed_df = (
    df.select(
        col("value").cast("string").alias("json_data"),
        "partition",
        "offset",
        "timestamp"
    )
)

In [0]:
ride_schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("rider_id", StringType(), True),
    StructField("pickup_lat", DoubleType(), True),
    StructField("pickup_lon", DoubleType(), True),
    StructField("drop_lat", DoubleType(), True),
    StructField("drop_lon", DoubleType(), True),
    StructField("fare", DoubleType(), True),
    StructField("distance", DoubleType(), True),
    StructField("ride_status", StringType(), True),
    StructField("event_time", TimestampType(), True)
])

rides = (
    parsed_df
        .select(
            "json_data",
            from_json(
                col("json_data"),
                ride_schema
            ).alias("data"),
            "partition",
            "offset",
            "timestamp"
        )
        .select(
            "json_data",
            "data.*",
            "partition",
            "offset",
            "timestamp"
        )
)

In [0]:
RUN_ID = str(py_uuid.uuid4())

landing_df = (
    rides
    .withColumn("landing_time", current_timestamp())
    .withColumn("pipeline_run_id", lit(RUN_ID))
    .withColumn("ingestion_source", lit("kafka"))
    .withColumn("kafka_topic", lit(TOPIC))
    .withColumn("kafka_partition", col("partition"))
    .withColumn("kafka_offset", col("offset"))
    .withColumn("kafka_timestamp", col("timestamp"))
    .withColumn("year", year("event_time"))
    .withColumn("month", month("event_time"))
    .withColumn("day", dayofmonth("event_time"))
    .withColumn("hour", hour("event_time"))
    .drop("partition", "offset", "timestamp")
)

In [0]:
query = (
    landing_df.writeStream
        .format("parquet")
        .partitionBy(
            "year",
            "month",
            "day"
        )
        .option(
            "path",
            "/Volumes/workspace/default/landing/ride_events"
        )
        .option(
            "checkpointLocation",
            "/Volumes/workspace/default/checkpoints/kafka_to_landing"
        )
         .trigger(availableNow=True)
        # .trigger(processingTime="30 seconds")
        .start()
)

In [0]:
query.exception()